# Barbet Harness（Colab 版）

驗證 Barbet 在 A／B／C 三種設定下的任務完成能力。

> ⚠️ 預設使用 **DUMMY 假模型**，結果不是 Barbet 的結果。取得主線的 checkpoint、renderer、解碼設定後，再到第 5 步切換。

## 1. 取得專案
從 GitHub 下載最新版本，直接執行即可。

In [ ]:
import os
if not os.path.exists('barbet-harness'):
    !git clone -q https://github.com/cathy910207-hub/barbet-harness.git
%cd barbet-harness
!git pull -q
!ls

## 2. 匯入 Tülu 資料集（規則執行題）
下載主線指定的固定版本（revision `fe0c7d35…`），抽出 23 題「每條規則都能解析、也都能用程式檢查」的題目。公開資料集不需要 Hugging Face token。

In [ ]:
!pip -q install huggingface_hub pandas pyarrow
!python importers/tulu_import.py --n 23

### 下載原始資料檔（給 Claude 校正解析器用）
Claude 的工作環境無法連到 Hugging Face。執行這格會下載固定版本的原始檔，請把下載的 `tulu_train.parquet` 傳回對話中。

In [ ]:
import glob, shutil
from huggingface_hub import snapshot_download
from google.colab import files
local = snapshot_download('allenai/tulu-3-sft-personas-instruction-following', repo_type='dataset',
                          revision='fe0c7d350c9b4542b8d829a6f1daa1c259f0ba0e', allow_patterns=['*.parquet'])
src = sorted(glob.glob(f'{local}/**/*.parquet', recursive=True))
print(src)
shutil.copy(src[0], 'tulu_train.parquet')
files.download('tulu_train.parquet')

## 3. 驗證資料
逐題用程式獨立重算或重查答案，產生 `data/candidates.jsonl`、`data/rejected.jsonl`、`data/manifest.json`。

In [ ]:
!python verify.py

## 4. 測試驗證器本身
故意做壞的資料、違反規則的回答，一定要被抓出來。

In [ ]:
!python tests/test_rule_checks.py
!python tests/test_tulu_parse.py
!python tests/test_verify.py
!python tests/test_tulu_integration.py

## 5. 檢查流程
應顯示「全部檢查通過 ✓」。

In [ ]:
!python tests/test_pipeline.py

## 6. 用假模型劇本跑 A／B／C
劇本在 `tests/dummy_script.json`，用來確認各條路徑（修正成功、修正後變錯、編造來源……）都能被正確記錄。

In [ ]:
!python run_experiment.py --dummy-script tests/dummy_script.json

## 7. 查看報告與 trace

In [ ]:
import glob, json
from IPython.display import Markdown, display
run_dir = sorted(glob.glob('runs/*'))[-1]
display(Markdown(open(f'{run_dir}/summary.md', encoding='utf-8').read()))

In [ ]:
# 看某一題、某一組的完整 trace
TASK, GROUP = 'fix_001', 'C'
for line in open(f'{run_dir}/traces.jsonl', encoding='utf-8'):
    r = json.loads(line)
    if r['task_id'] == TASK and r['group'] == GROUP:
        for s in r['steps']:
            print('='*20, f"第 {s['step']+1} 步", '='*20)
            print(s['prompt'][-600:])
            print('--- 模型輸出 ---\n', s['output'])
            if s.get('tool_execution'): print('--- 工具結果 ---\n', s['tool_execution'])
            if s.get('runtime_error'): print('--- 回饋 ---\n', s['runtime_error']['message'])
        print('\n最終：', '成功' if r['success'] else f"失敗（{r['error_type']}）")

## 8. 切換到 Barbet（取得主線設定後）

1. 在 `config.json` 填入 `model.type = "barbet"`、`model.checkpoint`、`model.revision`、`model.label`
2. 填入 `tokenizer.name_or_path`
3. 把 `decoding` 改成主線指定的設定
4. 在 `harness/renderer.py` 加入主線的 renderer，並修改 `renderer.name`

先只跑一題確認可以運作：

In [ ]:
# !pip install -q transformers torch
# !python run_experiment.py --tasks doc_001 --groups A

## 9. 記憶位置與無關段落實驗
修改 `config.json` 的 `retrieval.position`（start／middle／end）與 `retrieval.num_distractors`，各跑一次並比較。

In [ ]:
import json, subprocess
RUN = False  # 接上 Barbet 後改成 True 才會真的執行
cfg = json.load(open('config.json', encoding='utf-8'))
for pos in ['start', 'middle', 'end']:
    cfg['retrieval'].update(position=pos, num_distractors=3)
    cfg['experiment_name'] = f'position_{pos}'
    json.dump(cfg, open(f'config_{pos}.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
    if RUN:
        subprocess.run(['python', 'run_experiment.py', '--config', f'config_{pos}.json', '--groups', 'B,C'])
print('已產生 config_start/middle/end.json')

## 10. 下載交付檔案
包含 candidates／rejected／manifest、verify.py、tests、report.md 與執行紀錄。

In [ ]:
from google.colab import files
!zip -qr deliverables.zip data report.md verify.py tests runs
files.download('deliverables.zip')